In [17]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from datetime import datetime

from utils.plots import LivePlotCallback
from utils.preprocess_data import get_garbage_datasets
from utils.model_architectures import build_experimental_model
from config import IMG_SIZE, BATCH_SIZE, CLASSES, SEED, EPOCHS
import certifi

In [ ]:
# Tell SSL to use the certifi certificate bundle
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()

In [12]:
image_categories_path = '../data/images'

In [ ]:
train_ds, val_ds, class_names = get_garbage_datasets(
    data_dir=image_categories_path, 
    batch_size=BATCH_SIZE, 
    img_size=IMG_SIZE,
    model_type='mobilenet' 
)

Found 13901 files belonging to 6 classes.
Using 11121 files for training.
Found 13901 files belonging to 6 classes.
Using 2780 files for validation.


/Users/kvbek/Documents/projects/garbage-sorting-model/src/utils/model_architectures.py:101: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(


Model: "arch_mobilenet_v2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip_2 (RandomFlip)      │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation_2               │ (None, 256, 256, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom_2 (RandomZoom)      │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 8, 8, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 6)              │         1,542 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,588,486 (9.87 MB)

 Trainable params: 329,990 (1.26 MB)

 Non-trainable params: 2,258,496 (8.62 MB)

In [ ]:
# %%
architectures_to_test = [
    # 'control', 
    # 'shallow', 
    # 'vgg_style', 
    # 'large_kernel'
    'mobilenet_v2'
]

# Setup experiment directories (Stepping up one level to keep 'src' clean)
model_dir = '../models/experiments/'
log_dir = '../logs/experiments/'
os.makedirs(model_dir, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)

experiment_histories = {}

for arch in architectures_to_test:
    print(f"\n{'='*50}")
    print(f"🚀 STARTING EXPERIMENT: {arch.upper()}")
    print(f"{'='*50}\n")
    
    # 1. Build the specific model
    model = build_experimental_model(arch)
    
    # 2. Set up unique callbacks for this architecture
    model_filepath = os.path.join(model_dir, f'{arch}_best.weights.h5') # Fix for Mac Freeze
    csv_log_file = os.path.join(log_dir, f'{arch}_history.csv')
    
    checkpoint = tf.keras.callbacks.ModelCheckpoint(
        filepath=model_filepath, 
        monitor='val_accuracy', 
        save_best_only=True, 
        save_weights_only=True, # Critical for Mac
        verbose=1
    )
    
    lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy', factor=0.5, patience=3, min_lr=0.00001, verbose=1
    )
    
    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=7, restore_best_weights=True, verbose=1
    )
    
    csv_logger = tf.keras.callbacks.CSVLogger(csv_log_file, append=False)
    
    # 3. Train the model
    history = model.fit(
        train_ds, 
        validation_data=val_ds, 
        epochs=EPOCHS, # Uses your config variable
        callbacks=[checkpoint, lr_scheduler, early_stop, csv_logger],
        verbose=1 
    )
    
    # Store history in memory for immediate plotting
    experiment_histories[arch] = history.history

In [5]:
# Download the model
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(256, 256, 3),
    include_top=False, # CRITICAL: This chops off Google's 1000-class layer!
    weights='imagenet' # Load the pre-trained brain
)

# FREEZE THE BRAIN
base_model.trainable = False 

/var/folders/k0/426ngj994pqf16x0xd47lrx40000gn/T/ipykernel_4848/3344113243.py:2: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = tf.keras.applications.MobileNetV2(


In [ ]:
model = tf.keras.Sequential([
    # 1. Define the input
    tf.keras.layers.Input(shape=(256, 256, 3)),
    
    # 2. Add the frozen MobileNetV2 brain
    base_model,
    
    # 3. Add your custom head (just like your previous model)
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(len(CLASSES), activation='softmax')
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 8, 8, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │         1,542 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,588,486 (9.87 MB)

 Trainable params: 329,990 (1.26 MB)

 Non-trainable params: 2,258,496 (8.62 MB)

In [7]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_path = '../models/net_v2/'
os.makedirs(model_path, exist_ok=True)
model_name = f'best_garbage_classifier_{datetime.now().strftime("%Y-%m-%d_%H-%M-%S")}.weights.h5'
model_filepath = os.path.join(model_path, model_name)

# 1. Save the best model automatically
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=model_filepath, 
    monitor='val_accuracy',                   
    mode='max',                               
    save_best_only=True,
    save_weights_only=True, # CRITICAL FIX for Mac Freezing
    verbose=1                                 
)

# 2. Lower learning rate if stuck for 3 epochs
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_accuracy', 
    factor=0.5,      
    patience=3,      
    min_lr=0.00001,  
    verbose=1
)

# 3. Stop training entirely if stuck for 7 epochs
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy', 
    patience=7,                   # Wait 7 epochs before stopping
    restore_best_weights=True,    # Automatically reload the best weights at the end
    verbose=1
)

# # Create an instance
live_plot = LivePlotCallback()

# 4. Start training (Epochs set to 50, but it will likely stop earlier!)
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50, 
    callbacks=[
        checkpoint_callback, 
        lr_scheduler, 
        early_stopping
    ] 
)

Epoch 1/50


2026-05-22 20:02:14.340582: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


174/174 ━━━━━━━━━━━━━━━━━━━━ 0s 156ms/step - accuracy: 0.6786 - loss: 0.9859
Epoch 1: val_accuracy improved from None to 0.84604, saving model to ../models/net_v2/best_garbage_classifier_2026-05-22_20-02-14.weights.h5

Epoch 1: finished saving model to ../models/net_v2/best_garbage_classifier_2026-05-22_20-02-14.weights.h5
174/174 ━━━━━━━━━━━━━━━━━━━━ 39s 206ms/step - accuracy: 0.7615 - loss: 0.7116 - val_accuracy: 0.8460 - val_loss: 0.4717 - learning_rate: 0.0010
Epoch 2/50
174/174 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step - accuracy: 0.8379 - loss: 0.4775
Epoch 2: val_accuracy improved from 0.84604 to 0.85072, saving model to ../models/net_v2/best_garbage_classifier_2026-05-22_20-02-14.weights.h5

Epoch 2: finished saving model to ../models/net_v2/best_garbage_classifier_2026-05-22_20-02-14.weights.h5
174/174 ━━━━━━━━━━━━━━━━━━━━ 27s 152ms/step - accuracy: 0.8363 - loss: 0.4742 - val_accuracy: 0.8507 - val_loss: 0.4711 - learning_rate: 0.0010
Epoch 3/50
174/174 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms

In [8]:
# 1. Create the logs directory if it doesn't exist yet
os.makedirs('logs', exist_ok=True)

# 2. Convert the history dictionary to a Pandas DataFrame
history_df = pd.DataFrame(history.history)

# 3. Save it as a CSV (index=False prevents adding an extra column of row numbers)
history_df.to_csv('logs/mobile_net_v2_history.csv', index=False)

print("Training history saved to logs/mobile_net_v2_history.csv")

Training history saved to logs/mobile_net_v2_history.csv
